# Sentinel-1 and Sentinel-2 U-Net Example

In [ ]:
!pip install -q geemap ee_extra tensorflow
import ee, geemap
ee.Initialize()

In [ ]:
# Authenticate with Earth Engine
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

In [ ]:
# Define Region of Interest (ROI)
roi = ee.FeatureCollection('projects/ee-liaons/assets/rebio_gurupi')

In [ ]:
def maskS2clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
        qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).divide(10000)

In [ ]:
start_date = '2020-01-01'
end_date = '2020-12-31'
s2 = (ee.ImageCollection('COPERNICUS/S2_SR')
        .filterBounds(roi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(maskS2clouds)
        .median())

In [ ]:
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(roi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .select(['VV','VH'])
        .median())

In [ ]:
combo = s2.addBands(s1)

In [ ]:
Map = geemap.Map(center=[-3.8, -47.5], zoom=8)
Map.addLayer(combo, {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 0.3
}, 'Sentinel RGB')
Map.addLayer(s1.select('VV'), {'min': -20, 'max': 0}, 'Sentinel1 VV')
Map

In [ ]:
task = ee.batch.Export.image.toDrive(**{
    'image': combo,
    'description': 's1_s2_combo',
    'folder': 'GEE_exports',
    'scale': 10,
    'region': roi.geometry(),
    'maxPixels': 1e13
})
task.start()

## Deep Learning (U-Net) Training
Use TensorFlow/Keras to train a U-Net model on the exported patches.
This part mirrors the logic in `deep_learning_crop.R`.
Prepare image patches, create a U-Net architecture, train on your dataset, and evaluate results.